# 📬 Notebook 4: Queues and Load Shedding

When traffic spikes exceed capacity, use queues to absorb bursts and load shedding to gracefully degrade.

## Learning Objectives

By the end of this notebook, you'll understand:
- Using queues to absorb write bursts
- Async write patterns
- Load shedding strategies
- Real-world examples (Uber, Strava)

In [1]:
import redis
import json
import time
import random
from datetime import datetime
from typing import Optional, Callable

r = redis.Redis(host='localhost', port=6379, decode_responses=True)
r.flushall()

print("✅ Connected to Redis!")
print("📊 Open RedisInsight: http://localhost:5540")

✅ Connected to Redis!
📊 Open RedisInsight: http://localhost:5540


## 📬 Write Queues

In [2]:
print("📬 Queue-Based Write Pattern")
print("=" * 60)
print("""
PROBLEM: Bursty traffic overwhelms database
─────────────────────────────────────────────────────────────
              BURST!                    
    Writers ━━━━━━━━━━━━━━━━━━━━━━> [Database] 💥 Overloaded!
              10,000 writes/sec          Max: 1,000/sec

─────────────────────────────────────────────────────────────

SOLUTION: Queue absorbs bursts, workers drain at safe rate
─────────────────────────────────────────────────────────────
              BURST!           Steady drain
    Writers ━━━━━━━> [Queue] ━━━━━━━━━━━━━> [Database] ✓
              10K/sec   Buffer    1K/sec      Happy!

• Queue grows during bursts
• Queue drains during calm periods
• Database sees steady write rate
""")

📬 Queue-Based Write Pattern

PROBLEM: Bursty traffic overwhelms database
─────────────────────────────────────────────────────────────
              BURST!                    
    Writers ━━━━━━━━━━━━━━━━━━━━━━> [Database] 💥 Overloaded!
              10,000 writes/sec          Max: 1,000/sec

─────────────────────────────────────────────────────────────

SOLUTION: Queue absorbs bursts, workers drain at safe rate
─────────────────────────────────────────────────────────────
              BURST!           Steady drain
    Writers ━━━━━━━> [Queue] ━━━━━━━━━━━━━> [Database] ✓
              10K/sec   Buffer    1K/sec      Happy!

• Queue grows during bursts
• Queue drains during calm periods
• Database sees steady write rate



In [3]:
class WriteQueue:
    def __init__(self, queue_name: str, max_size: int = 10000):
        self.queue_name = queue_name
        self.max_size = max_size
        self.stats = {"enqueued": 0, "dropped": 0, "processed": 0}
    
    def enqueue(self, data: dict) -> bool:
        current_size = r.llen(self.queue_name)
        if current_size >= self.max_size:
            self.stats["dropped"] += 1
            return False
        
        data["queued_at"] = datetime.now().isoformat()
        r.rpush(self.queue_name, json.dumps(data))
        self.stats["enqueued"] += 1
        return True
    
    def dequeue(self, batch_size: int = 1) -> list:
        items = []
        for _ in range(batch_size):
            item = r.lpop(self.queue_name)
            if item:
                items.append(json.loads(item))
                self.stats["processed"] += 1
            else:
                break
        return items
    
    def size(self) -> int:
        return r.llen(self.queue_name)

print("📊 Simulating Bursty Traffic")
print("=" * 60)

queue = WriteQueue("location_updates", max_size=1000)

print("\n🌊 Simulating burst: 500 writes in 'rush hour'")
for i in range(500):
    queue.enqueue({"user_id": i, "lat": 40.7 + random.random(), "lng": -74.0 + random.random()})

print(f"   Queue size: {queue.size()}")
print(f"   Enqueued: {queue.stats['enqueued']}")
print(f"   Dropped: {queue.stats['dropped']}")

print("\n⏱️ Simulating steady processing...")
while queue.size() > 0:
    batch = queue.dequeue(batch_size=50)
    time.sleep(0.01)

print(f"   Processed: {queue.stats['processed']}")
print("\n✅ Database saw steady 50 writes/batch instead of 500 spike!")

📊 Simulating Bursty Traffic

🌊 Simulating burst: 500 writes in 'rush hour'
   Queue size: 500
   Enqueued: 500
   Dropped: 0

⏱️ Simulating steady processing...


   Processed: 500

✅ Database saw steady 50 writes/batch instead of 500 spike!


## 🚮 Load Shedding

In [4]:
print("🚮 Load Shedding Strategies")
print("=" * 60)
print("""
When queue is full, strategically DROP less important writes.

STRATEGIES:
─────────────────────────────────────────────────────────────

1. DROP OLDEST (Uber location updates)
   ┌────────────────────────────────┐
   │ Old │ Old │ Old │ New │ New │  │ ← New
   └──────▲─────────────────────────┘
          └── Drop stale locations

2. DROP NEWEST (Preserve order)
   ┌────────────────────────────────┐
   │ 1st │ 2nd │ 3rd │ 4th │ 5th │ │ ← Reject new
   └────────────────────────────────┘
       First come, first served

3. DROP BY PRIORITY
   ┌──────────────────────────────────────┐
   │ P1! │ P1! │ P2  │ P3  │ P3  │ P3 │  │
   └─────────────────────▲────────────────┘
                         └── Drop low priority first

4. SAMPLE (Strava segments)
   Keep every Nth location point:
   ✓ • • ✓ • • ✓ • • ✓ • • 
""")

🚮 Load Shedding Strategies

When queue is full, strategically DROP less important writes.

STRATEGIES:
─────────────────────────────────────────────────────────────

1. DROP OLDEST (Uber location updates)
   ┌────────────────────────────────┐
   │ Old │ Old │ Old │ New │ New │  │ ← New
   └──────▲─────────────────────────┘
          └── Drop stale locations

2. DROP NEWEST (Preserve order)
   ┌────────────────────────────────┐
   │ 1st │ 2nd │ 3rd │ 4th │ 5th │ │ ← Reject new
   └────────────────────────────────┘
       First come, first served

3. DROP BY PRIORITY
   ┌──────────────────────────────────────┐
   │ P1! │ P1! │ P2  │ P3  │ P3  │ P3 │  │
   └─────────────────────▲────────────────┘
                         └── Drop low priority first

4. SAMPLE (Strava segments)
   Keep every Nth location point:
   ✓ • • ✓ • • ✓ • • ✓ • • 



In [5]:
class PriorityQueue:
    def __init__(self):
        self.queue_name = "priority_writes"
        self.stats = {"high": 0, "medium": 0, "low": 0, "dropped": 0}
    
    def enqueue(self, data: dict, priority: str = "medium") -> bool:
        score = {"high": 3, "medium": 2, "low": 1}[priority]
        r.zadd(self.queue_name, {json.dumps(data): score})
        self.stats[priority] += 1
        return True
    
    def shed_load(self, keep_count: int):
        total = r.zcard(self.queue_name)
        if total > keep_count:
            to_remove = total - keep_count
            r.zpopmin(self.queue_name, to_remove)
            self.stats["dropped"] += to_remove
            return to_remove
        return 0

print("🎯 Priority-Based Load Shedding")
print("=" * 60)

pq = PriorityQueue()
r.delete(pq.queue_name)

for i in range(30):
    pq.enqueue({"id": i, "type": "low_priority"}, "low")
for i in range(20):
    pq.enqueue({"id": i, "type": "medium_priority"}, "medium")
for i in range(10):
    pq.enqueue({"id": i, "type": "high_priority"}, "high")

print(f"\n📊 Before shedding:")
print(f"   High priority: {pq.stats['high']}")
print(f"   Medium priority: {pq.stats['medium']}")
print(f"   Low priority: {pq.stats['low']}")
print(f"   Total in queue: {r.zcard(pq.queue_name)}")

dropped = pq.shed_load(keep_count=25)
print(f"\n🚮 After shedding (keep top 25):")
print(f"   Dropped: {dropped} low-priority items")
print(f"   Remaining: {r.zcard(pq.queue_name)}")

print("\n✅ High priority writes preserved during overload!")

🎯 Priority-Based Load Shedding

📊 Before shedding:
   High priority: 10
   Medium priority: 20
   Low priority: 30
   Total in queue: 60

🚮 After shedding (keep top 25):
   Dropped: 35 low-priority items
   Remaining: 25

✅ High priority writes preserved during overload!


## 🚗 Real World: Uber Location Updates

In [6]:
print("🚗 Uber-Style Location Updates")
print("=" * 60)
print("""
Uber receives millions of GPS updates per second.
Not all updates are equally important!

Strategy: Only keep LATEST location per driver
─────────────────────────────────────────────────────────────

Driver 123 sends:
  10:00:01 → Lat: 40.7128  ← Overwritten
  10:00:02 → Lat: 40.7130  ← Overwritten  
  10:00:03 → Lat: 40.7132  ← KEPT (latest)

Result: 3 updates → 1 write to database!
""")

🚗 Uber-Style Location Updates

Uber receives millions of GPS updates per second.
Not all updates are equally important!

Strategy: Only keep LATEST location per driver
─────────────────────────────────────────────────────────────

Driver 123 sends:
  10:00:01 → Lat: 40.7128  ← Overwritten
  10:00:02 → Lat: 40.7130  ← Overwritten  
  10:00:03 → Lat: 40.7132  ← KEPT (latest)

Result: 3 updates → 1 write to database!



In [7]:
class LatestOnlyQueue:
    def __init__(self, prefix: str = "driver_loc"):
        self.prefix = prefix
        self.updates_received = 0
        self.unique_keys = set()
    
    def update_location(self, driver_id: int, lat: float, lng: float):
        key = f"{self.prefix}:{driver_id}"
        r.hset(key, mapping={
            "lat": lat,
            "lng": lng,
            "updated_at": datetime.now().isoformat()
        })
        self.updates_received += 1
        self.unique_keys.add(key)
    
    def get_stats(self) -> dict:
        return {
            "updates_received": self.updates_received,
            "unique_drivers": len(self.unique_keys),
            "write_reduction": f"{(1 - len(self.unique_keys)/self.updates_received)*100:.1f}%"
        }

print("🚗 Simulating Driver Location Updates")
print("=" * 60)

location_queue = LatestOnlyQueue()

print("\n📍 Each of 100 drivers sends 50 location updates...")
for driver_id in range(100):
    base_lat, base_lng = 40.7 + random.random() * 0.1, -74.0 + random.random() * 0.1
    for update in range(50):
        lat = base_lat + random.random() * 0.001
        lng = base_lng + random.random() * 0.001
        location_queue.update_location(driver_id, lat, lng)

stats = location_queue.get_stats()
print(f"\n📊 Results:")
print(f"   Updates received: {stats['updates_received']}")
print(f"   Unique drivers: {stats['unique_drivers']}")
print(f"   Write reduction: {stats['write_reduction']}")

print("\n✅ 5,000 updates → 100 database writes!")

🚗 Simulating Driver Location Updates

📍 Each of 100 drivers sends 50 location updates...



📊 Results:
   Updates received: 5000
   Unique drivers: 100
   Write reduction: 98.0%

✅ 5,000 updates → 100 database writes!


## ☠️ Dead-Letter Queues & Backpressure

Real queue-based systems need two more ideas beyond "enqueue + drain":

### Dead-Letter Queue (DLQ)

If a worker fails to process an item **N times in a row** (e.g., a malformed
payload, a bug, a downstream outage), park it in a separate **dead-letter
queue** instead of retrying forever. A human or a repair job can inspect
and replay those items later.

```
main queue ---> worker --X fail 3x ---> dead-letter queue
                                        (triage later)
```

Without a DLQ, one poison message can block the whole pipeline.

### Backpressure

When the queue is growing faster than workers drain it, you're heading
for trouble. Healthy systems push that pressure **back to the producer**:

- **Reject with `429 Too Many Requests`** — the client retries with backoff.
- **Slow down the producer** — e.g., an SDK throttles its own calls.
- **Shed load** (previous section) — drop less important writes.

The golden rule: **an unbounded queue is a bug**, not a feature. It just
hides the overload until memory runs out.


In [ ]:
print("☠️ Simulating a dead-letter queue")
print("=" * 60)

r.delete("work_queue", "dlq")
MAX_RETRIES = 3

def enqueue_work(item: dict, retries: int = 0):
    item["retries"] = retries
    r.rpush("work_queue", json.dumps(item))

# Producer: 10 items, item #4 is poisoned (always fails)
for i in range(10):
    enqueue_work({"id": i, "poisoned": i == 4})

def process(item: dict):
    if item.get("poisoned"):
        raise RuntimeError("cannot parse payload")

processed, dead = 0, 0
while r.llen("work_queue") > 0:
    raw = r.lpop("work_queue")
    item = json.loads(raw)
    try:
        process(item)
        processed += 1
    except Exception as e:
        if item["retries"] + 1 >= MAX_RETRIES:
            r.rpush("dlq", json.dumps({"item": item, "error": str(e)}))
            dead += 1
        else:
            enqueue_work(item, retries=item["retries"] + 1)

print(f"\n✅ Processed: {processed}")
print(f"☠️ Sent to DLQ: {dead}")
print(f"   DLQ contents: {r.lrange('dlq', 0, -1)}")


## 🧪 Quick Quiz

1. **When should you drop oldest vs newest items?**

2. **Why does Uber only keep the latest location per driver?**

3. **What's the trade-off of queue-based writes?**

In [8]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Drop oldest vs newest:")
print("   Drop OLDEST: Real-time data (locations, metrics)")
print("   Drop NEWEST: Order matters (transactions, logs)")
print()
print("2. Uber keeps latest only because:")
print("   - Old locations are stale/useless")
print("   - Only current position matters for matching")
print("   - Reduces writes by 50x or more")
print()
print("3. Queue trade-offs:")
print("   Pros: Absorbs bursts, steady DB load")
print("   Cons: Eventual consistency (delay)")
print("         Queue failure = data loss")

📝 Quiz Answers

1. Drop oldest vs newest:
   Drop OLDEST: Real-time data (locations, metrics)
   Drop NEWEST: Order matters (transactions, logs)

2. Uber keeps latest only because:
   - Old locations are stale/useless
   - Only current position matters for matching
   - Reduces writes by 50x or more

3. Queue trade-offs:
   Pros: Absorbs bursts, steady DB load
   Cons: Eventual consistency (delay)
         Queue failure = data loss


## 📚 Summary

### Key Takeaways

1. **Queues absorb bursts** - Buffer writes during spikes
2. **Load shedding** - Strategically drop less important writes
3. **Latest-only** - For real-time data, overwrite instead of append
4. **Priority queues** - Preserve important writes under load
5. **Trade-off** - Eventual consistency for better availability

### Next Up

In **Notebook 5**, we'll learn about batching and aggregation:
- Combining multiple writes into one
- Hierarchical counters
- Like/view aggregation patterns